In [ ]:
# ============================================================
# Camada Silver - Domínio UF
# ============================================================
# Objetivo:
# Transformar bronze.uf em três tabelas Silver:
# 1. silver.dim_uf
# 2. silver.fato_resultado_uf
# 3. silver.fato_distribuicao_nivel_uf
# ============================================================

from pathlib import Path
from datetime import date
import pandas as pd

BRONZE_PATH = Path("../data/bronze")
SILVER_PATH = Path("../data/silver")
EXECUTION_DATE = date.today().isoformat()

print("Bronze path:", BRONZE_PATH.resolve())
print("Silver path:", SILVER_PATH.resolve())
print("Execution date:", EXECUTION_DATE)


# ------------------------------------------------------------
# Função: obter_arquivo_mais_recente
# ------------------------------------------------------------
# Esta função procura arquivos Parquet dentro da pasta de uma
# tabela da camada Bronze e retorna o arquivo mais recente.
#
# Ela é útil porque os dados estão particionados por data de
# execução, por exemplo:
#
# data/bronze/uf/execution_date=2026-06-29/uf.parquet
#
# Assim, quando houver mais de uma execução, o notebook sempre
# irá carregar a versão mais recente disponível.
# ------------------------------------------------------------

def obter_arquivo_mais_recente(caminho_tabela: Path) -> Path:
    arquivos = list(caminho_tabela.rglob("*.parquet"))

    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo Parquet encontrado em: {caminho_tabela}")

    return max(arquivos, key=lambda arquivo: arquivo.stat().st_mtime)


# ------------------------------------------------------------
# Função: padronizar_texto
# ------------------------------------------------------------
# Esta função padroniza campos textuais simples.
#
# O objetivo é remover espaços extras no início e no fim dos
# valores, sem alterar acentuação ou caixa do texto.
#
# Exemplo:
# " Amazonas " -> "Amazonas"
# ------------------------------------------------------------

def padronizar_texto(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip()


# ------------------------------------------------------------
# Função: padronizar_sigla_uf
# ------------------------------------------------------------
# Esta função padroniza a sigla da UF.
#
# O objetivo é garantir que a sigla esteja:
# - como texto;
# - sem espaços extras;
# - em letras maiúsculas.
#
# Exemplo:
# " sp " -> "SP"
# ------------------------------------------------------------

def padronizar_sigla_uf(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip().upper()


# ------------------------------------------------------------
# Função: validar_percentual
# ------------------------------------------------------------
# Esta função cria colunas de validação para campos percentuais.
#
# Para cada coluna informada, ela cria uma nova flag booleana:
#
# flag_nome_da_coluna_valido
#
# A regra aplicada é:
# - valores entre 0 e 100 são válidos;
# - valores nulos também são considerados válidos neste momento.
#
# Os nulos são aceitos porque, na Silver, ainda não vamos imputar
# ou substituir valores sem entendimento de negócio. A decisão de
# tratamento será documentada posteriormente.
# ------------------------------------------------------------

def validar_percentual(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    df_validado = df.copy()

    for coluna in colunas:
        if coluna in df_validado.columns:
            df_validado[f"flag_{coluna}_valido"] = (
                df_validado[coluna].isna()
                | df_validado[coluna].between(0, 100)
            )

    return df_validado


# ------------------------------------------------------------
# Função: salvar_silver
# ------------------------------------------------------------
# Esta função salva um DataFrame tratado na camada Silver.
#
# A estrutura de saída segue o padrão:
#
# data/silver/{nome_tabela}/execution_date=YYYY-MM-DD/{nome_tabela}.parquet
#
# Esse padrão mantém histórico por data de processamento e ajuda
# a organizar as camadas do data lake no formato medalhão.
# ------------------------------------------------------------

def salvar_silver(df: pd.DataFrame, nome_tabela: str) -> Path:
    output_dir = SILVER_PATH / nome_tabela / f"execution_date={EXECUTION_DATE}"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{nome_tabela}.parquet"

    df.to_parquet(output_file, index=False)

    print(f"[OK] {nome_tabela} salvo em: {output_file}")
    print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")

    return output_file


# ------------------------------------------------------------
# 1. Carregar bronze.uf
# ------------------------------------------------------------

arquivo_uf = obter_arquivo_mais_recente(BRONZE_PATH / "uf")
df_uf_bronze = pd.read_parquet(arquivo_uf)

print("Arquivo bronze.uf carregado:", arquivo_uf)
print("Dimensão bronze.uf:", df_uf_bronze.shape)

display(df_uf_bronze.head())


# ------------------------------------------------------------
# 2. Padronização base da tabela uf
# ------------------------------------------------------------
# Nesta etapa criamos uma cópia da Bronze e aplicamos tratamentos
# mínimos de Silver:
#
# - ajuste de tipo do ano;
# - padronização da sigla da UF;
# - limpeza de espaços em textos;
# - conversão de indicadores para numérico;
# - remoção de duplicidades exatas.
# ------------------------------------------------------------

df_uf_base = df_uf_bronze.copy()

df_uf_base["ano"] = df_uf_base["ano"].astype("Int64")
df_uf_base["sigla_uf"] = df_uf_base["sigla_uf"].apply(padronizar_sigla_uf)
df_uf_base["sigla_uf_nome"] = df_uf_base["sigla_uf_nome"].apply(padronizar_texto)
df_uf_base["serie"] = df_uf_base["serie"].apply(padronizar_texto)
df_uf_base["rede"] = df_uf_base["rede"].apply(padronizar_texto)

colunas_numericas = [
    "taxa_alfabetizacao",
    "media_portugues",
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
]

for coluna in colunas_numericas:
    if coluna in df_uf_base.columns:
        df_uf_base[coluna] = pd.to_numeric(df_uf_base[coluna], errors="coerce")

qtd_antes = len(df_uf_base)
df_uf_base = df_uf_base.drop_duplicates()
qtd_depois = len(df_uf_base)

print("Duplicidades exatas removidas:", qtd_antes - qtd_depois)


# ------------------------------------------------------------
# 3. Criar silver.dim_uf
# ------------------------------------------------------------
# A dimensão de UF representa o cadastro territorial da Unidade
# Federativa.
#
# Ela remove repetição de sigla e nome da UF das tabelas fato.
#
# Campos:
# - sigla_uf
# - sigla_uf_nome
# - data_processamento_silver
# ------------------------------------------------------------

df_dim_uf = (
    df_uf_base[
        [
            "sigla_uf",
            "sigla_uf_nome",
        ]
    ]
    .drop_duplicates()
    .sort_values("sigla_uf")
    .reset_index(drop=True)
)

df_dim_uf["data_processamento_silver"] = EXECUTION_DATE

print("Dimensão silver.dim_uf:", df_dim_uf.shape)
display(df_dim_uf.head(30))


# ------------------------------------------------------------
# 4. Criar silver.fato_resultado_uf
# ------------------------------------------------------------
# Esta tabela fato armazena os principais indicadores de
# alfabetização por:
#
# - ano;
# - UF;
# - série;
# - rede.
#
# Indicadores mantidos:
# - taxa_alfabetizacao;
# - media_portugues.
#
# A chave lógica esperada é:
# ano + sigla_uf + serie + rede
# ------------------------------------------------------------

df_fato_resultado_uf = df_uf_base[
    [
        "ano",
        "sigla_uf",
        "serie",
        "rede",
        "taxa_alfabetizacao",
        "media_portugues",
    ]
].copy()

df_fato_resultado_uf = validar_percentual(
    df_fato_resultado_uf,
    ["taxa_alfabetizacao"]
)

df_fato_resultado_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_resultado_uf = (
    df_fato_resultado_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "serie", "rede"])
    .reset_index(drop=True)
)

print("Dimensão silver.fato_resultado_uf:", df_fato_resultado_uf.shape)
display(df_fato_resultado_uf.head(20))


# ------------------------------------------------------------
# 5. Criar silver.fato_distribuicao_nivel_uf
# ------------------------------------------------------------
# A Bronze possui a distribuição por nível em várias colunas:
#
# proporcao_aluno_nivel_0
# proporcao_aluno_nivel_1
# ...
# proporcao_aluno_nivel_8
#
# Na Silver, vamos normalizar essa estrutura, transformando as
# colunas em linhas.
#
# Isso facilita:
# - filtros por nível;
# - gráficos;
# - comparações;
# - agregações;
# - modelagem analítica posterior na Gold.
#
# Exemplo de saída:
# ano | sigla_uf | serie | rede | nivel_alfabetizacao | proporcao_alunos
# ------------------------------------------------------------

colunas_niveis = [
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
]

df_fato_distribuicao_nivel_uf = df_uf_base.melt(
    id_vars=[
        "ano",
        "sigla_uf",
        "serie",
        "rede",
    ],
    value_vars=colunas_niveis,
    var_name="nivel_alfabetizacao",
    value_name="proporcao_alunos"
)

df_fato_distribuicao_nivel_uf["nivel_alfabetizacao"] = (
    df_fato_distribuicao_nivel_uf["nivel_alfabetizacao"]
    .str.extract(r"(\d+)")
    .astype("Int64")
)

df_fato_distribuicao_nivel_uf = validar_percentual(
    df_fato_distribuicao_nivel_uf,
    ["proporcao_alunos"]
)

df_fato_distribuicao_nivel_uf["data_processamento_silver"] = EXECUTION_DATE

df_fato_distribuicao_nivel_uf = (
    df_fato_distribuicao_nivel_uf
    .drop_duplicates()
    .sort_values(["ano", "sigla_uf", "serie", "rede", "nivel_alfabetizacao"])
    .reset_index(drop=True)
)

print("Dimensão silver.fato_distribuicao_nivel_uf:", df_fato_distribuicao_nivel_uf.shape)
display(df_fato_distribuicao_nivel_uf.head(30))


# ------------------------------------------------------------
# 6. Validações rápidas
# ------------------------------------------------------------
# As validações abaixo servem para conferir se a transformação
# gerou dados coerentes antes de salvar.
#
# São verificadas:
# - quantidade de UFs;
# - anos disponíveis;
# - redes;
# - séries;
# - níveis de alfabetização;
# - existência de percentuais inválidos;
# - concentração de nulos por ano.
# ------------------------------------------------------------

print("Validações - dim_uf")
print("UFs distintas:", df_dim_uf["sigla_uf"].nunique(dropna=True))
print("Linhas dim_uf:", len(df_dim_uf))

print("\nValidações - fato_resultado_uf")
print("Linhas:", len(df_fato_resultado_uf))
print("Anos:", sorted(df_fato_resultado_uf["ano"].dropna().unique().tolist()))
print("Redes:", df_fato_resultado_uf["rede"].dropna().unique().tolist())
print("Séries:", df_fato_resultado_uf["serie"].dropna().unique().tolist())
print("Flags taxa inválida:", (~df_fato_resultado_uf["flag_taxa_alfabetizacao_valido"]).sum())

print("\nValidações - fato_distribuicao_nivel_uf")
print("Linhas:", len(df_fato_distribuicao_nivel_uf))
print("Níveis:", sorted(df_fato_distribuicao_nivel_uf["nivel_alfabetizacao"].dropna().unique().tolist()))
print("Flags proporção inválida:", (~df_fato_distribuicao_nivel_uf["flag_proporcao_alunos_valido"]).sum())

print("\nPercentual de nulos em proporcao_alunos por ano:")
display(
    df_fato_distribuicao_nivel_uf
    .groupby("ano")["proporcao_alunos"]
    .apply(lambda x: round(x.isna().mean() * 100, 2))
    .reset_index(name="percentual_nulos_proporcao_alunos")
)


# ------------------------------------------------------------
# 7. Salvar tabelas Silver
# ------------------------------------------------------------
# Nesta etapa as três tabelas tratadas são gravadas em Parquet
# dentro da camada Silver.
#
# Os arquivos ficam apenas no ambiente local e não devem ser
# versionados no GitHub.
# ------------------------------------------------------------

arquivo_dim_uf = salvar_silver(df_dim_uf, "dim_uf")
arquivo_fato_resultado_uf = salvar_silver(df_fato_resultado_uf, "fato_resultado_uf")
arquivo_fato_distribuicao_nivel_uf = salvar_silver(
    df_fato_distribuicao_nivel_uf,
    "fato_distribuicao_nivel_uf"
)


# ------------------------------------------------------------
# 8. Conferência final
# ------------------------------------------------------------
# Após salvar, fazemos uma leitura dos próprios arquivos Parquet
# gerados para confirmar se eles foram gravados corretamente.
# ------------------------------------------------------------

for arquivo in [
    arquivo_dim_uf,
    arquivo_fato_resultado_uf,
    arquivo_fato_distribuicao_nivel_uf,
]:
    df_conferencia = pd.read_parquet(arquivo)
    print(f"{arquivo.name}: {df_conferencia.shape}")